# 01 — Bronze Layer Raw Data Storage (Databricks PySpark & Delta Lake)
- **Workstream 2:** Lakehouse & Data Quality (Member 2 — Ahmed Essam)
- **ClickUp Task:** `123u8tync4t` (*Implement Bronze raw-data storage*)
- **Sources Ingested:**
  1. `SRC-001`: openFDA Drug NDC (`openfda_ndc_*.json` — 100 records)
  2. `SRC-002`: openFDA Drug Enforcement Recalls (`openfda_enforcement_*.json` — 100 records)
  3. `SRC-007`: WHO Essential Medicines List 2023 (`who_eml_raw.json` or `bronze_who_eml.parquet` — 69 pages)

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

try:
    dbutils.widgets.text("volume_path", "/Volumes/workspace/default/raw_data", "Raw Volume Path")
    VOLUME_PATH = dbutils.widgets.get("volume_path").rstrip("/")
except Exception:
    VOLUME_PATH = "/Volumes/workspace/default/raw_data"

def resolve_latest_file(volume_path: str, pattern: str) -> str:
    matches = sorted(Path(volume_path).glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching '{pattern}' found in {volume_path}")
    return str(matches[-1])

# 1. Bronze openFDA NDC Table
ndc_path = resolve_latest_file(VOLUME_PATH, "openfda_ndc_*.json")
print(f"Reading NDC snapshot: {ndc_path}")
raw_ndc = spark.read.option("multiLine", "true").json(ndc_path)

bronze_ndc_df = (
    raw_ndc.select(
        F.col("metadata.source_id").alias("source_id"),
        F.col("metadata.source_name").alias("source_name"),
        F.col("metadata.source_endpoint").alias("source_endpoint"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.col("metadata.run_id").alias("run_id"),
        F.explode("data.results").alias("drug")
    )
    .select("source_id", "source_name", "source_endpoint", "ingestion_timestamp_utc", "run_id", "drug.*")
    .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

(
    bronze_ndc_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_openfda_ndc")
)
print(f"bronze_openfda_ndc rows: {bronze_ndc_df.count()}")
display(bronze_ndc_df.limit(5))

In [ ]:
# 2. Bronze openFDA Enforcement Recalls Table
enf_path = resolve_latest_file(VOLUME_PATH, "openfda_enforcement_*.json")
print(f"Reading Enforcement snapshot: {enf_path}")
raw_enf = spark.read.option("multiLine", "true").json(enf_path)

bronze_enf_df = (
    raw_enf.select(
        F.col("metadata.source_id").alias("source_id"),
        F.col("metadata.source_name").alias("source_name"),
        F.col("metadata.source_endpoint").alias("source_endpoint"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.col("metadata.run_id").alias("run_id"),
        F.explode("data.results").alias("recall")
    )
    .select("source_id", "source_name", "source_endpoint", "ingestion_timestamp_utc", "run_id", "recall.*")
    .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

(
    bronze_enf_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_openfda_enforcement")
)
print(f"bronze_openfda_enforcement rows: {bronze_enf_df.count()}")
display(bronze_enf_df.limit(5))

In [ ]:
# 3. Bronze WHO Essential Medicines List (2023) Table
who_json = Path(VOLUME_PATH) / "who_eml_raw.json"
who_parquet = Path(VOLUME_PATH) / "bronze_who_eml.parquet"

if who_json.exists():
    print(f"Reading WHO EML JSON: {who_json}")
    raw_who = spark.read.option("multiLine", "true").json(str(who_json))
    bronze_who_df = (
        raw_who.select(
            F.lit("SRC-007").alias("source_id"),
            F.col("metadata.source").alias("source_name"),
            F.col("metadata.source_file").alias("source_file"),
            F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
            F.explode("results").alias("page_rec")
        )
        .select("source_id", "source_name", "source_file", "ingestion_timestamp_utc", "page_rec.*")
        .withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
        .withColumn("bronze_loaded_at_utc", F.current_timestamp())
    )
elif who_parquet.exists():
    print(f"Reading WHO EML Parquet snapshot: {who_parquet}")
    bronze_who_df = spark.read.parquet(str(who_parquet))
    if "raw_record_sha256" not in bronze_who_df.columns:
        bronze_who_df = bronze_who_df.withColumn("raw_record_sha256", F.sha2(F.to_json(F.struct("*")), 256))
    bronze_who_df = bronze_who_df.withColumn("bronze_loaded_at_utc", F.current_timestamp())
else:
    raise FileNotFoundError(f"Neither who_eml_raw.json nor bronze_who_eml.parquet found in {VOLUME_PATH}")

(
    bronze_who_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_who_eml")
)
print(f"bronze_who_eml rows: {bronze_who_df.count()}")
display(bronze_who_df.limit(5))